# 14. 평가 고치기 (v2.1): LLM-as-a-Judge + 검증 생성셋

F1(단어 겹침)은 답변 품질을 과소평가합니다. "검색은 성공했는데 F1 < 0.4"인 답 154개 중 40개를 직접 판정하니 **정답 24 · 부분 9 · 오답 7**이었습니다.
그래서 앞으로 모든 실험(SFT v2, DPO 등)을 같은 기준으로 재기 위해 평가를 새로 만듭니다.

| 할 일 | 내용 |
|---|---|
| ① 검증 생성셋 | 검증 답 있음 200 (03의 `rerank/val.jsonl`) + **답 없음 100** (train에서, 06·13 학습에 안 쓴 질문). 질문 목록을 저장해서 이후 학습에서 제외 |
| ② 검증셋 답변 생성 | 08과 같은 챗봇 설정 (MiniLM 상위 5, 점수 < 4.92 거절, Qwen2.5-7B ± 06 LoRA) |
| ③ LLM 심판 | **Qwen2.5-14B-Instruct** (생성 모델보다 큰 모델). **정답 점수** 0/1/2 (질문 + 참고 정답 + 답변) · **근거 충실도** 0/1 (문단 + 답변: 모든 내용이 문단에 있나) |
| ④ 심판 검증 | 사람이 판정한 40개(`human40.jsonl`)와 일치율 |
| ⑤ 적용 | 시험(08 답변 650) + 검증(300) |

- 심판 결과는 `judge_eval/judgments.jsonl`에 (질문, 답변)별로 저장 → 이후 노트북에서 같은 답은 다시 채점 안 함
- A100 (40GB 이상) 기준 약 30분

## 1. 설치 + 경로

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu peft
!pip -q uninstall -y torchao   # 최신 peft가 Colab 기본 torchao(0.10)와 충돌해서 제거 (이 프로젝트엔 필요 없음)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
RAG_DIR    = f'{SAVE_DIR}/rag'                          # 08 결과 (시험 답변)
LORA_DIR   = f'{SAVE_DIR}/lora_sentence/adapter'        # 06 문장형 LoRA
OUT_DIR    = f'{SAVE_DIR}/judge_eval'
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True)
print('사람 판정 40개 있음:', os.path.exists(f'{OUT_DIR}/human40.jsonl'))

## 2. 설정 + 공통 함수

In [ ]:
import re, json, gzip, time, random, string, hashlib
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch

SEED      = 42
K, TOP_N  = 100, 5
THRESH    = 4.921875
GEN_LLM   = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_LLM = 'Qwen/Qwen2.5-14B-Instruct'
EMB_MODEL = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
NO_ANSWER = 'No Answer Present.'
N_VAL_NOANS = 100
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower(); s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref); c = sum((Counter(p) & Counter(r)).values())
    if not p or not r or c == 0:
        return 0.0
    pr, rc = c / len(p), c / len(r)
    return 2 * pr * rc / (pr + rc)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

SYS_RAG = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences."""

corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h
sites  = [site_of(c['url']) for c in corpus]

## 3. 검증 생성셋 만들기

- 답 있음: 03의 검증 200 (평가셋과 안 겹침)
- 답 없음: train 건강 질문 중 "No Answer Present"이고 문단이 DB에 있는 것에서, **06 LoRA 학습(450)·13 판정 모델 학습(4,000)에 쓴 질문을 빼고** 100개
- 질문 목록을 `valgen_set.jsonl`로 저장 → SFT v2·DPO 학습에서 제외

In [ ]:
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_ans  = [dict(q, group='A') for q in load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl') if q.get('gold_pids')]
held_out = {q['qid'] for q in eval_set} | {q['qid'] for q in val_ans}
pid_of   = {t: i for i, t in enumerate(texts)}

wf_pool, noans06, pos13, neg13 = [], [], [], []
for q in load_jsonl(f'{SAVE_DIR}/train_health_qa.jsonl.gz'):
    if not q['in_corpus'] or q['qid'] in held_out:
        continue
    gold = [pid_of[p['text']] for p in q['passages'] if p['is_selected'] == 1 and p['text'] in pid_of]
    if q['answers'] == [NO_ANSWER]:
        noans06.append(q['qid']); neg13.append(q)
    else:
        if q['well_formed'] and gold:
            wf_pool.append(q['qid'])
        if gold:
            pos13.append(q['qid'])
# 06과 같은 순서로 섞어서 06 거절 학습 질문 재현
r06 = random.Random(SEED); r06.shuffle(wf_pool); r06.shuffle(noans06)
used06 = set(noans06[:450])
# 13과 같은 순서로 섞어서 13 학습 질문 재현
r13 = random.Random(SEED); r13.shuffle(pos13); r13.shuffle(neg13)
used13 = {q['qid'] for q in neg13[:4000]}
pool = [q for q in neg13 if q['qid'] not in used06 | used13]
random.Random(7).shuffle(pool)
val_noans = [{'qid': q['qid'], 'query': q['query'], 'answers': [], 'gold_pids': [], 'group': 'N'} for q in pool[:N_VAL_NOANS]]
valgen = val_ans + val_noans
save_jsonl([{'qid': q['qid'], 'query': q['query'], 'group': q['group']} for q in valgen], f'{OUT_DIR}/valgen_set.jsonl')
print(f'검증 생성셋: 답 있음 {len(val_ans)} + 답 없음 {len(val_noans)} (남은 답 없음 후보 {len(pool):,}, 06·13 사용분 제외)')

## 4. 검증셋 검색 + 답변 생성 (08과 같은 설정)

In [ ]:
VAL_ANS_PATH = f'{OUT_DIR}/val_answers.jsonl'
if os.path.exists(VAL_ANS_PATH):
    val_rows = load_jsonl(VAL_ANS_PATH)
    print('저장된 검증 답변 불러옴')
else:
    import faiss
    from sentence_transformers import SentenceTransformer, CrossEncoder
    index = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
    emb_model = SentenceTransformer(EMB_MODEL, device='cuda').half()
    reranker = CrossEncoder(RERANKER, max_length=256, device='cuda'); reranker.model.half()
    norm_text = lambda t: ' '.join(t.lower().split())
    queries = [q['query'] for q in valgen]
    qe = emb_model.encode([QUERY_PREFIX + q for q in queries], batch_size=128, normalize_embeddings=True,
                          convert_to_numpy=True).astype('float32')
    _, ids = index.search(qe, K)
    sc = reranker.predict([(q, texts[p]) for q, c in zip(queries, ids.tolist()) for p in c], batch_size=512).reshape(len(queries), K)
    val_rows = []
    for q, cands, s_ in zip(valgen, ids.tolist(), sc):
        docs, seen = [], set()
        for i in np.argsort(-s_):
            p = int(cands[i]); key = norm_text(texts[p])
            if key in seen:
                continue
            seen.add(key); docs.append({'pid': p, 'score': float(s_[i]), 'site': sites[p]})
            if len(docs) == TOP_N:
                break
        val_rows.append({**q, 'rag_docs': docs})
    del emb_model, reranker, index; torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    tok = AutoTokenizer.from_pretrained(GEN_LLM, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(GEN_LLM, dtype=torch.bfloat16, device_map='cuda')
    llm = PeftModel.from_pretrained(llm, LORA_DIR).eval()
    def rag_messages(q, docs):
        ctx = '\n'.join(f'[{i}] ({d["site"]}) {texts[d["pid"]]}' for i, d in enumerate(docs, 1))
        return [{'role': 'system', 'content': SYS_RAG}, {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]
    @torch.inference_mode()
    def generate(msgs_list, lora, bs=16):
        outs = []
        ctx = llm.disable_adapter() if not lora else __import__('contextlib').nullcontext()
        with ctx:
            for i in range(0, len(msgs_list), bs):
                prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
                enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
                gen = llm.generate(**enc, max_new_tokens=256, do_sample=False, temperature=None, top_p=None, top_k=None,
                                   pad_token_id=tok.pad_token_id)
                outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        return [o.strip() for o in outs]
    todo = [r for r in val_rows if r['rag_docs'][0]['score'] >= THRESH]
    for name, lora in [('RAG 개선', False), ('RAG 개선 + LoRA', True)]:
        t = time.time()
        outs = generate([rag_messages(r['query'], r['rag_docs']) for r in todo], lora)
        for r in val_rows:
            r[name] = REFUSAL
        for r, a in zip(todo, outs):
            r[name] = a
        print(f'{name}: {len(todo)}개 생성 {time.time() - t:.0f}초 (점수 기준 거절 {len(val_rows) - len(todo)})')
    save_jsonl(val_rows, VAL_ANS_PATH)
    del llm, tok; import gc; gc.collect(); torch.cuda.empty_cache()   # 7B를 GPU에서 확실히 내림

## 5. LLM 심판 (Qwen2.5-14B-Instruct)

답을 생성하지 않고 첫 토큰 확률만 봅니다 (빠르고 재현 가능).
- **정답 점수**: 0 오답 / 1 부분 / 2 정답 → 기대값 (0~2). 참고 정답과 **핵심 사실이 같으면 표현이 달라도 정답**
- **근거 충실도**: 답변의 모든 사실이 문단에 있으면 1 → P(1)
- 거절한 답은 심판 없이: 답 있는 질문이면 정답 점수 0

In [ ]:
import gc; gc.collect(); torch.cuda.empty_cache()
print(f'심판 불러오기 전 GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB')
from transformers import AutoTokenizer, AutoModelForCausalLM
jtok = AutoTokenizer.from_pretrained(JUDGE_LLM, padding_side='left')
judge = AutoModelForCausalLM.from_pretrained(JUDGE_LLM, dtype=torch.bfloat16, device_map='cuda').eval()
DIGITS = [jtok.encode(d, add_special_tokens=False)[0] for d in '012']

SYS_CORRECT = ('You grade answers to health questions. Compare the candidate answer with the reference answer(s). '
               'Reply with one digit only.\n'
               '2 = correct: states the key facts of a reference answer (different wording or extra correct detail is fine)\n'
               '1 = partially correct: some key facts, but incomplete or partly wrong\n'
               '0 = wrong, unrelated, or does not answer the question')
SYS_FAITH = ('You check whether an answer is supported by the given documents. Reply with one digit only.\n'
             '1 = every factual statement in the answer is stated in the documents\n'
             '0 = the answer contains a fact that is not in the documents, or contradicts them')

def correct_msgs(q, refs, ans):
    ref = '\n'.join(f'- {x}' for x in refs[:3])
    return [{'role': 'system', 'content': SYS_CORRECT},
            {'role': 'user', 'content': f'Question: {q}\nReference answer(s):\n{ref}\nCandidate answer: {ans}\n\nGrade (0, 1 or 2):'}]

def faith_msgs(docs_text, ans):
    return [{'role': 'system', 'content': SYS_FAITH},
            {'role': 'user', 'content': f'Documents:\n{docs_text}\n\nAnswer: {ans}\n\nSupported (0 or 1)?'}]

@torch.inference_mode()
def digit_probs(msgs_list, n_digits, bs=8):
    prompts = [jtok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
    order = np.argsort([len(p) for p in prompts])          # 길이순으로 묶어서 빠르게
    out = [None] * len(prompts)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = jtok([prompts[i] for i in idx], return_tensors='pt', padding=True).to(judge.device)
        logits = judge(**enc).logits[:, -1, DIGITS[:n_digits]].float()
        for i, p in zip(idx, torch.softmax(logits, -1).cpu().numpy()):
            out[i] = p
    return out

JUDGE_PATH = f'{OUT_DIR}/judgments.jsonl'
cache = {j['key']: j for j in load_jsonl(JUDGE_PATH)} if os.path.exists(JUDGE_PATH) else {}
key_of = lambda kind, q, a: hashlib.md5(f'{kind}|{q}|{a}'.encode()).hexdigest()

def judge_all(items):
    # items: dict(q, refs, ans, docs_text) → 같은 순서로 (정답 점수 0~2 또는 None, 근거 충실도 0~1 또는 None)
    need_c, need_f = [], []
    for it in items:
        if is_refusal(it['ans']):
            continue
        if it.get('refs') and key_of('c', it['q'], it['ans']) not in cache:
            need_c.append(it)
        if it.get('docs_text') and key_of('f', it['q'], it['ans']) not in cache:
            need_f.append(it)
    t = time.time()
    for it, p in zip(need_c, digit_probs([correct_msgs(it['q'], it['refs'], it['ans']) for it in need_c], 3)):
        k = key_of('c', it['q'], it['ans']); cache[k] = {'key': k, 'v': float(p[1] + 2 * p[2])}
    for it, p in zip(need_f, digit_probs([faith_msgs(it['docs_text'], it['ans']) for it in need_f], 2)):
        k = key_of('f', it['q'], it['ans']); cache[k] = {'key': k, 'v': float(p[1])}
    save_jsonl(list(cache.values()), JUDGE_PATH)
    print(f'심판: 정답 {len(need_c)}개 + 근거 {len(need_f)}개 새로 채점 ({time.time() - t:.0f}초)')
    res = []
    for it in items:
        if is_refusal(it['ans']):
            res.append((0.0 if it.get('refs') else None, None))
        else:
            res.append((cache[key_of('c', it['q'], it['ans'])]['v'] if it.get('refs') else None,
                        cache[key_of('f', it['q'], it['ans'])]['v'] if it.get('docs_text') else None))
    return res

docs_text_of = lambda r: '\n'.join(f'[{i}] ({d["site"]}) {texts[d["pid"]]}' for i, d in enumerate(r['rag_docs'], 1))

## 6. 심판 검증: 사람이 판정한 40개와 비교

In [ ]:
human = load_jsonl(f'{OUT_DIR}/human40.jsonl')
evq = {q['qid']: q for q in eval_set}
hres = judge_all([{'q': h['query'], 'refs': evq[h['qid']]['answers'], 'ans': h['answer']} for h in human])
judge_cls = [int(round(c)) for c, _ in hres]
hum = [h['human'] for h in human]
agree3 = np.mean([a == b for a, b in zip(judge_cls, hum)])
agree2 = np.mean([(a >= 1) == (b >= 1) for a, b in zip(judge_cls, hum)])
print(f'사람 vs 심판 일치율: 3단계 {agree3:.2f} / 맞음(정답+부분) vs 오답 {agree2:.2f}')
print('혼동표 (행=사람, 열=심판)'); conf = pd.crosstab(pd.Series(hum, name='사람'), pd.Series(judge_cls, name='심판')); display(conf)
print('참고: 이 40개의 F1은 모두 0.4 미만 (F1로는 전부 "틀림")')

## 7. 시험(08 답변)과 검증에 적용

In [ ]:
def evaluate(rows, systems, has_docs):
    table = {}
    for name in systems:
        items = []
        for r in rows:
            refs = r['answers'] if r['group'] == 'A' else None
            items.append({'q': r['query'], 'refs': refs, 'ans': r[name],
                          'docs_text': docs_text_of(r) if has_docs.get(name, True) else None})
        res = judge_all(items)
        A = [(r, c) for r, (c, _) in zip(rows, res) if r['group'] == 'A']
        answered = [(r, f) for r, (_, f) in zip(rows, res) if f is not None and not is_refusal(r[name])]
        g = lambda k: [r for r in rows if r['group'] == k]
        table[name] = {
            '정답률 (심판 2점)':      np.mean([c >= 1.5 for _, c in A]),
            '정답+부분 (심판 ≥ 1점)': np.mean([c >= 0.5 for _, c in A]),
            '평균 점수 (0~2)':        np.mean([c for _, c in A]),
            'F1 (참고)':              np.mean([max(f1(r[name], x) for x in r['answers']) for r, _ in A]),
            '근거 충실도':            np.mean([f >= 0.5 for _, f in answered]) if answered else np.nan,
            '잘못 거절':              np.mean([is_refusal(r[name]) for r in g('A')]),
            '답 없음 거절률':         np.mean([is_refusal(r[name]) for r in g('N')]),
            '분야 밖 거절률':         np.mean([is_refusal(r[name]) for r in g('O')]) if g('O') else np.nan,
        }
    return pd.DataFrame(table).T.round(3)

def add_s1(rows):
    for r in rows:
        lora, base = r['RAG 개선 + LoRA'], r['RAG 개선']
        r['S1: LoRA 거절 시 원본'] = base if is_refusal(lora) else lora

test_rows = load_jsonl(f'{RAG_DIR}/eval_answers.jsonl')
for r in test_rows:
    q = evq[r['qid']]; r['answers'] = q['answers']
    r['group'] = 'A' if r['domain'] == 'health' and r['answerable'] else ('N' if r['domain'] == 'health' else 'O')
add_s1(test_rows)
test_tab = evaluate(test_rows, ['B0', 'RAG 기본', 'RAG 개선', 'RAG 개선 + LoRA', 'S1: LoRA 거절 시 원본'], {'B0': False})
print('[시험 650 (08 답변)]'); display(test_tab)

add_s1(val_rows)
val_tab = evaluate(val_rows, ['RAG 개선', 'RAG 개선 + LoRA', 'S1: LoRA 거절 시 원본'], {})
print('[검증 300 (답 있음 200 + 답 없음 100)]'); display(val_tab)

In [ ]:
# 심판이 '정답'인데 F1은 낮은 예 / 심판이 '오답'인 예 (발표용)
A = [r for r in test_rows if r['group'] == 'A' and not is_refusal(r['RAG 개선 + LoRA'])]
def cval(r):
    return cache[key_of('c', r['query'], r['RAG 개선 + LoRA'])]['v']
hi = sorted([r for r in A if cval(r) >= 1.5], key=lambda r: max(f1(r['RAG 개선 + LoRA'], x) for x in r['answers']))[:3]
lo = [r for r in A if cval(r) < 0.5][:3]
for title, rs in [('심판 정답 · F1 낮음', hi), ('심판 오답', lo)]:
    print(f'\n== {title}')
    for r in rs:
        print(f"Q: {r['query']}\n  정답: {r['answers'][0][:110]}\n  답변: {r['RAG 개선 + LoRA'][:150]}  (심판 {cval(r):.2f})")

## 8. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
summary = '\n'.join([
    f'# 14 평가 고치기: LLM-as-a-Judge (v2.1) ({stamp} KST)', '',
    f'- 심판: {JUDGE_LLM} 첫 토큰 확률. 정답 점수 0/1/2 (참고 정답과 핵심 사실 비교), 근거 충실도 0/1 (답변의 모든 사실이 문단에 있나)',
    f'- 심판 검증 (사람 판정 40개, F1 < 0.4인 답): 3단계 일치 {agree3:.2f} / 맞음 vs 오답 일치 {agree2:.2f}',
    f'- 검증 생성셋: 답 있음 {len(val_ans)} + 답 없음 {len(val_noans)} (06·13 학습 질문 제외, 목록 `judge_eval/valgen_set.jsonl`)',
    '- 정답률 = 심판 2점(≥1.5) 비율, 거절은 0점. 근거 충실도 = 거절 안 한 답 중 문단에 다 있는 비율', '',
    '## 시험 650 (08 답변)', '', test_tab.to_markdown(), '',
    '## 검증 300', '', val_tab.to_markdown(), '',
    '## 심판 vs 사람 (행=사람, 열=심판)', '', conf.to_markdown(),
])
for path in (f'{OUT_DIR}/results_summary.md', f'{REPORT_DIR}/14_judge_eval_summary.md'):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(summary)
print(summary)

## 9. 드라이브 저장 마무리 + 런타임 종료

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()